# 01 — CNN Intel Image Classification

**Dataset**: Intel Image Classification (~25.000 gambar, 6 kelas: buildings, forest, glacier, mountain, sea, street)

---

## Cara Pakai

> **Jalankan semua cell dari atas ke bawah secara berurutan.**

### Alur Kerja
```
[Setup]    gpu-setup → colab-mount → colab-clone → fix-patches → path-setup
              ↓
[Bagian 1] Utility Functions (PIL/Pillow + NumPy)
              ↓
[Bagian 2] Forward Propagation From Scratch (Conv2D, LC2D, Pooling, Dense)
              ↓
[Bagian 3] Pelatihan Keras CNN
           ├── 3a: 16 variasi Conv2D
           └── 3b: 16 variasi LocallyConnected2D
              ↓
[Bagian 4] Evaluasi & Perbandingan
           ├── Conv2D vs LocallyConnected2D
           ├── Ranking semua konfigurasi
           ├── Scratch CNN vs Keras CNN
           ├── Confusion Matrix
           └── Feature Map Visualization (Bonus)
```

### Hasil Disimpan ke Drive
- Bobot model → `MyDrive/weights/cnn/*.h5`
- Hasil JSON  → `MyDrive/results/cnn/*.json`
- Plot        → `MyDrive/results/cnn/*.png`

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    print(f'[GPU] Ditemukan {len(gpus)} GPU: {[g.name for g in gpus]}')
    print(f'[TF]  Versi TensorFlow : {tf.__version__}')
    print(f'[TF]  Built with CUDA  : {tf.test.is_built_with_cuda()}')
else:
    print('[GPU] Tidak ada GPU terdeteksi — menggunakan CPU.')
    try:
        import google.colab
        print('      Di Colab: Runtime → Change runtime type → GPU (T4/L4), lalu Restart session.')
    except ImportError:
        print('      Di lokal: pastikan driver NVIDIA + tensorflow[and-cuda] sudah terinstall.')
    print(f'[TF]  Versi TensorFlow : {tf.__version__}')
    print(f'[TF]  Built with CUDA  : {tf.test.is_built_with_cuda()}')


In [ ]:
import sys

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('[Colab] Google Drive terpasang.')
else:
    print('[Lokal] Tidak di Colab — skip Drive mount.')


In [ ]:
import sys, os

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

REPO_URL = 'https://github.com/danenftyessir/ChosaHeidan_Tubes-2_IF3270.git'
REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'

if IN_COLAB:
    if not os.path.exists(REPO_DIR):
        os.system(f'git clone {REPO_URL} {REPO_DIR}')
    else:
        os.system(f'git -C {REPO_DIR} pull')
    print(f'[Repo] Kode tersedia di {REPO_DIR}/src/')
else:
    print('[Lokal] Skip clone.')


In [ ]:
# ── Fix relative imports (runtime patch, tidak perlu push ke repo) ─────────────
import os, re

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    _src = os.path.join(REPO_DIR, 'src')

    # 1. Buat __init__.py di semua package dir
    for _d in ['', 'shared', 'cnn', 'cnn/scratch', 'cnn/keras', 'cnn/utils', 'cnn/bonus',
               'lstm', 'lstm/scratch', 'lstm/keras', 'lstm/bonus',
               'rnn', 'rnn/scratch', 'rnn/keras', 'rnn/bonus']:
        _init = os.path.join(_src, _d, '__init__.py')
        if not os.path.exists(_init):
            open(_init, 'w').close()

    def _fix_rel(path, bare_import, make_block):
        with open(path, 'r') as f:
            c = f.read()
        c = re.sub(
            r'(?m)^([ ]*)try:[ ]*\n[ ]*' + re.escape(bare_import.lstrip()) +
            r'[ ]*\n[ ]*except ImportError:[ ]*\n[^\n]*',
            lambda m: m.group(1) + bare_import.lstrip(), c
        )
        c = re.sub(
            r'^([ ]*)' + re.escape(bare_import.lstrip()) + r'$',
            lambda m: make_block(m.group(1)), c, flags=re.MULTILINE
        )
        with open(path, 'w') as f:
            f.write(c)

    # 2. Fix shared/dense.py
    _fix_rel(
        os.path.join(_src, 'shared', 'dense.py'),
        'from .activations import get_activation',
        lambda ind: (f'{ind}try:\n'
                     f'{ind}    from .activations import get_activation\n'
                     f'{ind}except ImportError:\n'
                     f'{ind}    from activations import get_activation')
    )

    # 3. Fix shared/intel_preprocess.py
    _fix_rel(
        os.path.join(_src, 'shared', 'intel_preprocess.py'),
        'from ..cnn.utils.utils import load_image',
        lambda ind: (f'{ind}try:\n'
                     f'{ind}    from ..cnn.utils.utils import load_image\n'
                     f'{ind}except ImportError:\n'
                     f'{ind}    from cnn.utils.utils import load_image')
    )

    print('[Fix-01] Patches applied — CNN imports OK')
else:
    print('[Fix-01] Lokal — skip patch')


In [ ]:
import os, sys
import numpy as np

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    pass

DRIVE_INTEL_ID = '1Amb6Wi42esiEowKKAeEXalb22FHSEWne'

if IN_COLAB:
    REPO_DIR = '/content/ChosaHeidan_Tubes-2_IF3270'
    SRC_DIR  = os.path.join(REPO_DIR, 'src')
    _MYDRIVE = '/content/drive/MyDrive'

    DATA_DIR        = os.path.join(_MYDRIVE, 'intel_image_classification')
    CNN_WEIGHTS_DIR = os.path.join(_MYDRIVE, 'weights', 'cnn')
    CNN_RESULTS_DIR = os.path.join(_MYDRIVE, 'results', 'cnn')

    # Fallback: Drive API jika folder belum ter-mount
    if not os.path.exists(DATA_DIR):
        print('[Path] Folder intel tidak ditemukan di Drive mount — pakai Drive API...')
        from google.colab import auth
        auth.authenticate_user()
        from googleapiclient.discovery import build
        from googleapiclient.http import MediaIoBaseDownload
        import io, concurrent.futures

        _svc = build('drive', 'v3')

        def _dl_folder(folder_id, dest, workers=6):
            os.makedirs(dest, exist_ok=True)
            items, page_token = [], None
            while True:
                resp = _svc.files().list(
                    q=f"'{folder_id}' in parents and trashed=false",
                    fields='nextPageToken,files(id,name,mimeType)',
                    pageToken=page_token, pageSize=1000
                ).execute()
                items.extend(resp.get('files', []))
                page_token = resp.get('nextPageToken')
                if not page_token:
                    break
            files = [(i, os.path.join(dest, i['name'])) for i in items
                     if i['mimeType'] != 'application/vnd.google-apps.folder']
            dirs  = [(i, os.path.join(dest, i['name'])) for i in items
                     if i['mimeType'] == 'application/vnd.google-apps.folder']
            def _one(pair):
                item, path = pair
                if os.path.exists(path):
                    return
                req = _svc.files().get_media(fileId=item['id'])
                with io.FileIO(path, 'wb') as fh:
                    dl = MediaIoBaseDownload(fh, req, chunksize=8*1024*1024)
                    done = False
                    while not done:
                        _, done = dl.next_chunk()
            with concurrent.futures.ThreadPoolExecutor(max_workers=workers) as ex:
                list(ex.map(_one, files))
            for sub, sub_dest in dirs:
                _dl_folder(sub['id'], sub_dest, workers)

        print('[Drive API] Mengunduh intel_image_classification ...')
        _dl_folder(DRIVE_INTEL_ID, DATA_DIR)

else:
    SRC_DIR = os.path.abspath('.')
    if not os.path.exists(os.path.join(SRC_DIR, 'cnn')):
        SRC_DIR = os.path.join(os.path.abspath('.'), 'src')
    PROJECT_ROOT    = os.path.dirname(SRC_DIR)
    DATA_DIR        = os.path.join(PROJECT_ROOT, 'data', 'intel_image_classification')
    CNN_WEIGHTS_DIR = os.path.join(PROJECT_ROOT, 'weights', 'cnn')
    CNN_RESULTS_DIR = os.path.join(PROJECT_ROOT, 'results', 'cnn')

sys.path.insert(0, SRC_DIR)
sys.path.insert(0, os.path.join(SRC_DIR, 'shared'))

for d in [CNN_WEIGHTS_DIR, CNN_RESULTS_DIR]:
    os.makedirs(d, exist_ok=True)

print(f'[Path] SRC_DIR         : {SRC_DIR}  (exists: {os.path.exists(SRC_DIR)})')
print(f'[Path] DATA_DIR        : {DATA_DIR}  (exists: {os.path.exists(DATA_DIR)})')
print(f'[Path] CNN_WEIGHTS_DIR : {CNN_WEIGHTS_DIR}')
print(f'[Path] CNN_RESULTS_DIR : {CNN_RESULTS_DIR}')

if IN_COLAB:
    assert os.path.exists(os.path.join(DATA_DIR, 'seg_train')),         f'seg_train/ tidak ada di: {DATA_DIR}'
    assert os.path.exists(os.path.join(DATA_DIR, 'seg_test')),         f'seg_test/ tidak ada di: {DATA_DIR}'
    print('[Path] Semua path OK')


## Bagian 1 — Utility Functions (PIL/Pillow + NumPy)

Load gambar dari disk menggunakan PIL, kemudian konversi ke array NumPy.
Fungsi-fungsi ini dipakai oleh semua bagian lainnya.

In [ ]:
import numpy as np
from cnn.utils.utils import load_image
from shared.preprocessing import load_image as preprocess_load_image, load_batch

print('[B1] Utility Functions dari shared/preprocessing.py')

# ── Demo load_image ────────────────────────────────────────────────────────────
import os
sample_dir = os.path.join(DATA_DIR, 'seg_train', 'buildings')
sample_file = next(
    (os.path.join(sample_dir, f) for f in os.listdir(sample_dir)
     if f.lower().endswith('.jpg')),
    None
)

if sample_file:
    img = preprocess_load_image(sample_file, target_size=(150, 150))
    print(f'  load_image  : shape={img.shape}, dtype={img.dtype}, '
          f'range=[{img.min():.0f}, {img.max():.0f}]')

    # Batch loading
    batch_files = [
        os.path.join(sample_dir, f)
        for f in os.listdir(sample_dir)[:3]
        if f.lower().endswith('.jpg')
    ]
    batch = load_batch(batch_files, target_size=(150, 150))
    print(f'  load_batch  : shape={batch.shape}  (3 images stacked)')
else:
    print('  [WARN] Tidak ada gambar di seg_train/buildings/ — skip demo')


## Bagian 2 — Forward Propagation From Scratch (NumPy)

Demonstrasi forward pass melalui setiap layer CNN yang diimplementasikan dari nol:
- `Conv2D` — konvolusi dengan bobot bersama (weight sharing)
- `LocallyConnected2D` — konvolusi **tanpa** weight sharing (setiap posisi spasial punya bobot sendiri)
- `MaxPooling2D`, `AveragePooling2D`, `GlobalAveragePooling2D`
- `Flatten`
- `Dense` (fully connected)
- `CNNScratch` — model penuh yang menyusun semua layer di atas

In [ ]:
import numpy as np
from cnn.scratch.conv2d import Conv2D
from cnn.scratch.locally_connected2d import LocallyConnected2D
from cnn.scratch.pooling import MaxPooling2D, AveragePooling2D, GlobalAveragePooling2D
from cnn.scratch.flatten import Flatten
from cnn.scratch.model_scratch import CNNScratch
from shared.dense import Dense

print('[B2] Scratch layers berhasil diimport.')


In [ ]:
import numpy as np

# ── Demo CNNScratch dengan Conv2D ─────────────────────────────────────────────
print('='*60)
print('CNNScratch Forward Pass (Conv2D + Pooling + Dense)')
print('='*60)

np.random.seed(42)
x_demo = np.random.rand(2, 64, 64, 3).astype(np.float32)   # batch of 2 images

layers_conv = [
    Conv2D(filters=8, kernel_size=3, strides=(1,1), padding='same', activation='relu'),
    MaxPooling2D(pool_size=(2,2)),
    Conv2D(filters=16, kernel_size=3, strides=(1,1), padding='same', activation='relu'),
    AveragePooling2D(pool_size=(2,2)),
    GlobalAveragePooling2D(),
    Dense(input_dim=16, units=8, activation='softmax'),
]

model_conv = CNNScratch(layers=layers_conv, num_classes=8, input_shape=(64, 64, 3))
out_conv = model_conv.forward(x_demo, verbose=True)
print(f'Output shape: {out_conv.shape}  (batch=2, classes=8)')
print(f'Softmax sums: {out_conv.sum(axis=1)}  (harus ~1.0 per baris)')

# ── Demo LocallyConnected2D (ukuran kecil karena lambat) ──────────────────────
print()
print('='*60)
print('LocallyConnected2D Forward Pass (no weight sharing)')
print('='*60)

x_small = np.random.rand(1, 8, 8, 3).astype(np.float32)
lc_layer = LocallyConnected2D(filters=4, kernel_size=3, strides=(1,1), padding='valid', activation='relu')
# Inisialisasi bobot manual (C_in=3)
lc_out = lc_layer.forward(x_small)
print(f'  Input  : {x_small.shape}')
print(f'  Output : {lc_out.shape}  (H_out, W_out berubah karena valid padding)')
print(f'  Bobot  : {lc_layer.weights.shape}  '
      f'(H_out*W_out, kH*kW*C_in, C_out) — unik per posisi spasial')

# ── Demo Flatten ──────────────────────────────────────────────────────────────
print()
flat = Flatten()
flat_out = flat.forward(lc_out)
print(f'  Flatten output: {flat_out.shape}')


## Bagian 3 — Pelatihan CNN (Keras) dengan Variasi Hyperparameter

Training dilakukan dua kali:
1. **Conv2D** (weight sharing) — 16 kombinasi hyperparameter
2. **LocallyConnected2D** (tanpa weight sharing) — 16 kombinasi yang sama

| Hyperparameter | Nilai |
|---|---|
| Jumlah conv layer | 2, 4 |
| Jumlah filter | 32, 128 |
| Ukuran kernel | (3,3), (5,5) |
| Jenis pooling | max, average |

**Total**: 2 × 2 × 2 × 2 = **16 model per arsitektur**

> Bobot terbaik (best val_f1) tiap model otomatis disimpan ke Drive.
> Early stopping patience=5. Estimasi total waktu: ~2–3 jam di L4/A100 GPU.

In [ ]:
from shared.intel_preprocess import IntelImagePreprocessor, INTEL_CLASSES

preprocessor = IntelImagePreprocessor(DATA_DIR, target_size=(150, 150))
preprocessor.load_data()
preprocessor.summary()


### 3a — Training Conv2D (16 variasi)

Setiap konfigurasi dilatih dengan early stopping (patience=5, monitor=val_f1).
Output per epoch menampilkan loss, acc, val_loss, val_acc, val_f1, LR, waktu, dan ETA.

In [ ]:
import json as _json, os as _os
from cnn.keras.train import train_with_variations

_conv2d_json = _os.path.join(CNN_RESULTS_DIR, 'conv2d_variations.json')

if _os.path.exists(_conv2d_json):
    with open(_conv2d_json, encoding='utf-8') as _f:
        conv2d_results = _json.load(_f)
    print(f'[B3a] Cache ditemukan — skip training, {len(conv2d_results)} model dimuat dari JSON.')
else:
    print('[B3a] Training 16 variasi Conv2D...')
    conv2d_results = train_with_variations(
        data_dir=DATA_DIR,
        arch_type='conv2d',
        layer_variations=[2, 4],
        filter_variations=[32, 128],
        kernel_variations=[(3, 3), (5, 5)],
        pooling_variations=['max', 'average'],
        epochs=30,
        batch_size=32,
        weights_dir=CNN_WEIGHTS_DIR,
        results_path=_conv2d_json,
    )
    print(f'[B3a] Selesai — {len(conv2d_results)} model Conv2D dilatih.')


In [ ]:
# ── Ranking Conv2D berdasarkan best val_f1 ─────────────────────────────────────
ranked_conv2d = sorted(
    [(k, v) for k, v in conv2d_results.items() if 'best_val_f1' in v],
    key=lambda x: x[1]['best_val_f1'],
    reverse=True,
)

print(f'{"="*65}')
print('  RANKING Conv2D — Val Macro F1 (higher is better)')
print(f'{"="*65}')
for i, (name, res) in enumerate(ranked_conv2d, 1):
    mark = ' <-- BEST' if i == 1 else ''
    print(f'  {i:2d}. {name:<45} F1={res["best_val_f1"]:.4f}{mark}')

BEST_CONV2D_NAME = ranked_conv2d[0][0]
print(f'\n[B3a] Best Conv2D  : {BEST_CONV2D_NAME}')
print(f'[B3a] Bobot        : {CNN_WEIGHTS_DIR}/{BEST_CONV2D_NAME}.h5')


### 3b — Training LocallyConnected2D (16 variasi)

LocallyConnected2D berbeda dari Conv2D: setiap posisi spasial memiliki bobot tersendiri
(tidak di-share), sehingga parameter jauh lebih banyak tetapi berpotensi menangkap
pola lokal yang berbeda per posisi.

> **Perhatian**: LocallyConnected2D lebih lambat dari Conv2D. Estimasi waktu ~3–4× lebih lama.

In [ ]:
import json as _json, os as _os

_lc_json = _os.path.join(CNN_RESULTS_DIR, 'lc_variations.json')

if _os.path.exists(_lc_json):
    with open(_lc_json, encoding='utf-8') as _f:
        lc_results = _json.load(_f)
    print(f'[B3b] Cache ditemukan — skip training, {len(lc_results)} model dimuat dari JSON.')
else:
    print('[B3b] Training 16 variasi LocallyConnected2D...')
    lc_results = train_with_variations(
        data_dir=DATA_DIR,
        arch_type='locallyconnected',
        layer_variations=[2, 4],
        filter_variations=[32, 128],
        kernel_variations=[(3, 3), (5, 5)],
        pooling_variations=['max', 'average'],
        epochs=30,
        batch_size=32,
        weights_dir=CNN_WEIGHTS_DIR,
        results_path=_lc_json,
    )
    print(f'[B3b] Selesai — {len(lc_results)} model LocallyConnected2D dilatih.')


In [ ]:
# ── Ranking LocallyConnected2D ─────────────────────────────────────────────────
ranked_lc = sorted(
    [(k, v) for k, v in lc_results.items() if 'best_val_f1' in v],
    key=lambda x: x[1]['best_val_f1'],
    reverse=True,
)

print(f'{"="*65}')
print('  RANKING LocallyConnected2D — Val Macro F1')
print(f'{"="*65}')
for i, (name, res) in enumerate(ranked_lc, 1):
    mark = ' <-- BEST' if i == 1 else ''
    print(f'  {i:2d}. {name:<45} F1={res["best_val_f1"]:.4f}{mark}')

BEST_LC_NAME = ranked_lc[0][0]
print(f'\n[B3b] Best LC      : {BEST_LC_NAME}')
print(f'[B3b] Bobot        : {CNN_WEIGHTS_DIR}/{BEST_LC_NAME}.h5')


## Bagian 4 — Evaluasi & Perbandingan

### Evaluasi yang dilakukan:
1. **Conv2D vs LocallyConnected2D** — bandingkan F1, akurasi, dan jumlah parameter
2. **Semua konfigurasi** — ranking gabungan Conv2D + LC dari hasil JSON
3. **Scratch vs Keras** — muat bobot Keras ke model scratch NumPy, bandingkan prediksi
4. **Confusion Matrix** — heatmap prediksi vs ground truth untuk model terbaik
5. **Feature Map** (Bonus) — visualisasi aktivasi layer Conv pertama

In [ ]:
from cnn.keras.evaluate import compare_shared_vs_non_shared

print('[B4-1] Membandingkan Conv2D vs LocallyConnected2D pada test set...')

conv2d_h5 = os.path.join(CNN_WEIGHTS_DIR, f'{BEST_CONV2D_NAME}.h5')
lc_h5     = os.path.join(CNN_WEIGHTS_DIR, f'{BEST_LC_NAME}.h5')

arch_comparison = compare_shared_vs_non_shared(
    conv2d_weights=conv2d_h5,
    local_weights=lc_h5,
    data_dir=DATA_DIR,
    split='test',
    batch_size=32,
    class_names=INTEL_CLASSES,
    verbose=True,
)

# Ringkasan
c2 = arch_comparison.get('conv2d_results', {})
lc = arch_comparison.get('locallyconnected_results', {})
print(f'\n  {"Arsitektur":<25} {"Macro F1":>10} {"Accuracy":>10} {"#Params":>12}')
print(f'  {"-"*58}')
print(f'  {"Conv2D (shared)":<25} {c2.get("macro_f1",0):>10.4f} '
      f'{c2.get("accuracy",0):>10.4f} {c2.get("num_params",0):>12,}')
print(f'  {"LocallyConnected2D":<25} {lc.get("macro_f1",0):>10.4f} '
      f'{lc.get("accuracy",0):>10.4f} {lc.get("num_params",0):>12,}')


In [ ]:
from cnn.keras.evaluate import compare_multiple_configs
import json as _json

print('[B4-2] Ranking semua konfigurasi (Conv2D + LC) dari file JSON...')

# Gabungkan hasil Conv2D dan LC
all_results = {}
for path in [
    os.path.join(CNN_RESULTS_DIR, 'conv2d_variations.json'),
    os.path.join(CNN_RESULTS_DIR, 'lc_variations.json'),
]:
    if os.path.exists(path):
        with open(path) as f:
            all_results.update(_json.load(f))

if all_results:
    all_ranked = sorted(
        [(k, v) for k, v in all_results.items() if 'best_val_f1' in v],
        key=lambda x: x[1]['best_val_f1'],
        reverse=True,
    )
    print(f'\n  Total konfigurasi: {len(all_ranked)}')
    print(f'  {"Rank":<5} {"Config":<50} {"Val F1":>8}')
    print(f'  {"-"*65}')
    for i, (name, res) in enumerate(all_ranked, 1):
        print(f'  {i:<5} {name:<50} {res["best_val_f1"]:>8.4f}')
    print(f'\n  OVERALL BEST: {all_ranked[0][0]}  (F1={all_ranked[0][1]["best_val_f1"]:.4f})')
else:
    print('  [SKIP] File JSON belum ada — jalankan Bagian 3 terlebih dahulu.')


In [ ]:
from cnn.keras.evaluate import run_part4_evaluation

print('[B4-3] Scratch CNN vs Keras CNN — memuat bobot Keras ke model NumPy...')

part4_results = run_part4_evaluation(
    results_json_path=os.path.join(CNN_RESULTS_DIR, 'conv2d_variations.json'),
    weights_dir=CNN_WEIGHTS_DIR,
    data_dir=DATA_DIR,
    split='test',
    batch_size=32,
    class_names=INTEL_CLASSES,
    save_dir=CNN_RESULTS_DIR,
    verbose=True,
)

if part4_results:
    keras_f1   = part4_results.get('keras_macro_f1', 0)
    scratch_f1 = part4_results.get('scratch_macro_f1', 0)
    diff       = abs(keras_f1 - scratch_f1)
    print(f'\n  Keras  Macro F1 : {keras_f1:.4f}')
    print(f'  Scratch Macro F1: {scratch_f1:.4f}')
    print(f'  Selisih         : {diff:.6f}  (ideal: < 1e-4)')


In [ ]:
from cnn.keras.evaluate import evaluate_from_weights
from shared.plot_utils import plot_confusion_matrix
from shared.intel_preprocess import INTEL_CLASSES, load_intel_dataset

print('[B4-4] Confusion Matrix untuk model Conv2D terbaik pada test set...')

best_h5 = os.path.join(CNN_WEIGHTS_DIR, f'{BEST_CONV2D_NAME}.h5')

# Evaluasi untuk mendapatkan y_true dan y_pred
eval_results = evaluate_from_weights(
    model=None,
    weights_path=best_h5,
    data_dir=DATA_DIR,
    split='test',
    batch_size=32,
    class_names=INTEL_CLASSES,
    verbose=True,
)

if eval_results and 'y_true' in eval_results and 'y_pred' in eval_results:
    cm_save = os.path.join(CNN_RESULTS_DIR, 'confusion_matrix_best.png')
    plot_confusion_matrix(
        y_true=eval_results['y_true'],
        y_pred=eval_results['y_pred'],
        classes=INTEL_CLASSES,
        save_path=cm_save,
        normalize=True,
        title=f'Confusion Matrix — {BEST_CONV2D_NAME}',
    )
else:
    print('  [INFO] eval_results tidak mengandung y_true/y_pred — skip CM plot.')
    print(f'  Macro F1 : {eval_results.get("macro_f1", "N/A")}')
    print(f'  Accuracy : {eval_results.get("accuracy", "N/A")}')


In [ ]:
# ── Bagian 4-5 (Bonus) — Feature Map Visualization ────────────────────────────
from cnn.bonus.bonus_feature_maps import visualize_feature_maps, visualize_filter_weights
from shared.preprocessing import load_image as _load_img
import os, numpy as np

print('[B4-Bonus] Visualisasi feature maps dari layer Conv2D pertama...')

# Ambil 1 gambar sample dari test set
sample_dir = os.path.join(DATA_DIR, 'seg_test', 'buildings')
sample_files = [
    os.path.join(sample_dir, f)
    for f in os.listdir(sample_dir)[:1]
    if f.lower().endswith('.jpg')
]

if sample_files:
    img = _load_img(sample_files[0], target_size=(150, 150))
    img_norm = img.astype(np.float32) / 255.0

    fm_save = os.path.join(CNN_RESULTS_DIR, 'feature_maps_sample.png')
    fw_save = os.path.join(CNN_RESULTS_DIR, 'filter_weights_conv1.png')

    visualize_feature_maps(
        model_weights_path=os.path.join(CNN_WEIGHTS_DIR, f'{BEST_CONV2D_NAME}.h5'),
        image=img_norm,
        layer_index=0,
        save_path=fm_save,
        max_filters=16,
    )
    visualize_filter_weights(
        model_weights_path=os.path.join(CNN_WEIGHTS_DIR, f'{BEST_CONV2D_NAME}.h5'),
        layer_index=0,
        save_path=fw_save,
    )
else:
    print('  [SKIP] Tidak ada gambar di seg_test/buildings/')


### Bagian 4-Bonus-2 — GradCAM Visualization

**Gradient-weighted Class Activation Mapping (GradCAM)** menunjukkan area gambar
yang paling berpengaruh terhadap prediksi kelas tertentu.
Menggunakan gradien yang mengalir ke layer Conv terakhir untuk membuat heatmap.

In [ ]:
from cnn.bonus.bonus_gradcam import (
    visualize_gradcam, visualize_multiple_gradcam, get_gradcam_config
)
from tensorflow.keras.models import load_model as keras_load_model
from shared.preprocessing import load_image as _load_img
import numpy as np

print('[Bonus-GradCAM] Memuat model Keras terbaik...')
_h5 = os.path.join(CNN_WEIGHTS_DIR, f'{BEST_CONV2D_NAME}.h5')
keras_model_gcam = keras_load_model(_h5)

# Tampilkan layer Conv yang tersedia
print('\nLayer Conv yang bisa digunakan untuk GradCAM:')
gcam_config = get_gradcam_config(keras_model_gcam)
for item in gcam_config:
    print(f'  {item}')

# Ambil satu gambar per kelas dari seg_test
gcam_images, gcam_paths = [], []
for cls in INTEL_CLASSES:
    cls_dir = os.path.join(DATA_DIR, 'seg_test', cls)
    if not os.path.exists(cls_dir):
        continue
    sample = next(
        (os.path.join(cls_dir, f) for f in os.listdir(cls_dir)
         if f.lower().endswith('.jpg')), None
    )
    if sample:
        img = _load_img(sample, target_size=(150, 150)).astype(np.float32) / 255.0
        gcam_images.append(img)
        gcam_paths.append(sample)

print(f'\nSample gambar: {len(gcam_images)} (1 per kelas)')

# Visualisasi GradCAM tunggal (kelas buildings)
if gcam_images:
    gcam_single_save = os.path.join(CNN_RESULTS_DIR, 'gradcam_single.png')
    visualize_gradcam(
        model=keras_model_gcam,
        image=gcam_images[0],
        class_names=INTEL_CLASSES,
        save_path=gcam_single_save,
    )

    # Visualisasi multi-gambar
    gcam_multi_save = os.path.join(CNN_RESULTS_DIR, 'gradcam_multi.png')
    visualize_multiple_gradcam(
        model=keras_model_gcam,
        images=gcam_images,
        class_names=INTEL_CLASSES,
        n_cols=3,
        save_path=gcam_multi_save,
    )
    print(f'[Bonus-GradCAM] Plot disimpan ke {CNN_RESULTS_DIR}/')


## Bagian 5 (Bonus) — CNN Backward Pass & Training from Scratch

Demonstrasi **backward propagation** melalui layer CNN scratch (NumPy):
- `gradient_checker` — verifikasi gradien numerik vs analitik
- `train_step` — satu langkah forward + backward + update bobot (SGD/Adam)

> **Catatan**: Dijalankan pada subset kecil (2 batch) hanya untuk verifikasi.
> Training CNN penuh dari scratch (tanpa Keras) bisa menggunakan `train_step` secara iteratif.

In [ ]:
from cnn.bonus.bonus_backward import (
    gradient_checker, train_step, backward_pass, compute_loss_gradient
)
from cnn.scratch.conv2d import Conv2D as ScratchConv2D
from cnn.scratch.pooling import GlobalAveragePooling2D as ScratchGAP
from cnn.scratch.flatten import Flatten as ScratchFlatten
from cnn.scratch.model_scratch import CNNScratch as _CNNScratch
from shared.dense import Dense as ScratchDense
from shared.preprocessing import load_image as _load_img
import numpy as np

print('[Bonus-Backward] Membangun scratch CNN kecil untuk demo backward...')

# Model kecil agar gradient check tidak terlalu lama
_layers_bwd = [
    ScratchConv2D(filters=4, kernel_size=3, padding='same', activation='relu'),
    ScratchGAP(),
    ScratchDense(input_dim=4, units=6, activation='softmax'),
]
scratch_bwd = _CNNScratch(layers=_layers_bwd, num_classes=6, input_shape=(32, 32, 3))

# Buat data mini (2 gambar, 32x32)
np.random.seed(0)
X_mini = np.random.rand(2, 32, 32, 3).astype(np.float64) * 0.1
y_mini = np.array([0, 2])   # labels

# ── Demo gradient checker ─────────────────────────────────────────────────────
print('\n[Bonus-Backward] Gradient checker (verifikasi numerik vs analitik):')
print('  Ini membandingkan gradien backward dengan finite-difference.')
print('  Jika selisih < 1e-4 untuk semua parameter, implementasi backward BENAR.')
gradient_checker(
    model=scratch_bwd,
    X_sample=X_mini[:1],
    label_sample=y_mini[:1],
    epsilon=1e-5,
    verbose=True,
)

# ── Demo train_step ────────────────────────────────────────────────────────────
print('\n[Bonus-Backward] Demo train_step (forward + backward + Adam update):')
for step in range(3):
    loss = train_step(
        model=scratch_bwd,
        X_batch=X_mini,
        y_batch=y_mini,
        optimizer='adam',
        learning_rate=0.001,
    )
    print(f'  Step {step+1}: loss = {loss:.4f}')
print('[Bonus-Backward] Selesai.')


### Bagian 5-Bonus-2 — Batch Inference Benchmark (CNNScratch)

Mengukur **throughput** (gambar/detik) dan **latency** (ms/gambar) implementasi
scratch CNN pada berbagai ukuran batch.

In [ ]:
from cnn.bonus.bonus_batch_inference import BatchInferenceRunner, evaluate_model as scratch_eval_model
from cnn.scratch.model_scratch import CNNScratch as _CNNScratch2, build_cnn_from_config
import numpy as np, json as _json

print('[Bonus-Batch] Membangun CNNScratch dari bobot Keras terbaik...')

# Load config dari results JSON
_results_json = os.path.join(CNN_RESULTS_DIR, 'conv2d_variations.json')
if os.path.exists(_results_json):
    with open(_results_json) as _f:
        _all_res = _json.load(_f)
    _cfg = _all_res.get(BEST_CONV2D_NAME, {}).get('config', {})
else:
    _cfg = {}

# Build scratch model & load weights
_scratch_bench = build_cnn_from_config(
    config={'layers': _cfg.get('layers', [
        {'type': 'conv2d', 'filters': 32, 'kernel_size': 3, 'activation': 'relu'},
        {'type': 'maxpool', 'pool_size': 2},
        {'type': 'conv2d', 'filters': 64, 'kernel_size': 3, 'activation': 'relu'},
        {'type': 'globalavgpool'},
        {'type': 'dense', 'units': 6, 'activation': 'softmax'},
    ])},
    num_classes=6, input_shape=(150, 150, 3)
)
_best_h5 = os.path.join(CNN_WEIGHTS_DIR, f'{BEST_CONV2D_NAME}.h5')
if os.path.exists(_best_h5):
    _scratch_bench.load_weights_from_h5(_best_h5)
    print(f'  Bobot dimuat dari: {_best_h5}')

# ── Siapkan data benchmark (subset kecil dari test set) ───────────────────────
from shared.preprocessing import load_batch as _lb
from shared.intel_preprocess import load_intel_dataset as _lid
_test_paths, _test_labels = _lid(DATA_DIR, split='test')
N_BENCH = min(64, len(_test_paths))
_X_bench = _lb(_test_paths[:N_BENCH], target_size=(150, 150)).astype(np.float32) / 255.0
_y_bench = np.array(_test_labels[:N_BENCH])

# ── Jalankan benchmark ─────────────────────────────────────────────────────────
print(f'\n[Bonus-Batch] Benchmark pada {N_BENCH} gambar...')
runner = BatchInferenceRunner(model=_scratch_bench)
bench_results = runner.run(_X_bench, batch_sizes=[1, 4, 8, 16, 32])

# ── Evaluasi akurasi scratch model ────────────────────────────────────────────
print('\n[Bonus-Batch] Evaluasi scratch model:')
scratch_eval_model(_scratch_bench, _X_bench, _y_bench, batch_size=16, verbose=True)
